# Step 5b — LLM zero-shot ASP-relevance scoring (Qwen2.5-7B-Instruct)

Goal: produce a 4th anchor with **truly different signal** from SPECTER2/SciBERT/Ridge. An LLM does its scoring by *reasoning* over the prompt rather than by similarity in a learned embedding space, so the prediction errors should de-correlate from the existing transformer anchors.

**Why this might break the 0.71 plateau:**
- SPECTER2, SciBERT, and any BERT-class model fine-tuned on the same 2494 rows hit ~0.636 OOF — they have saturated the information in `title + abstract` at the feature-extraction level.
- An LLM with a few-shot prompt encoded with our hand-curated rubric *brings outside knowledge* about ASP, Answer Set Programming, neuro-symbolic AI, etc. that BERT-class encoders only have implicit access to.
- Expected correlation vs SPECTER2: **0.6-0.8** (much lower than 0.948 for SciBERT). That is real diversity.

**Approach (calibrated zero-shot scoring, not chain-of-thought):**
1. Build a system prompt with the L5 rubric + 5 train examples (one per label).
2. For each paper, get the next-token logits and slice them for the 5 digit tokens '1', '2', '3', '4', '5'.
3. Softmax those 5 logits → 5 probabilities.
4. Continuous score = `1*p1 + 2*p2 + 3*p3 + 4*p4 + 5*p5`.

This is much cleaner than asking the LLM to *generate* a number — we never read text output, we read distribution.

**Time budget on Colab T4** (free): ~10-15 min for 3090 papers with Qwen2.5-7B 4-bit quantised.
**Time on A100**: 3-5 min.

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade "transformers>=4.41" "accelerate>=0.30" "bitsandbytes>=0.43" pandas numpy scikit-learn

## 2. Upload data + load

In [ ]:
import pathlib, zipfile
from google.colab import files

WORK = pathlib.Path('/content/work')
DATA = WORK / 'data'
OUT = WORK / 'outputs'
RUN_DIR = OUT / 'llm_zeroshot'
for d in [WORK, DATA, OUT, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, content in uploaded.items():
    target = WORK / name
    target.write_bytes(content)
    if name.lower().endswith('.zip'):
        with zipfile.ZipFile(target) as zf:
            zf.extractall(DATA)
        print('Extracted', name, '->', DATA)
for p in sorted(DATA.glob('*')):
    print(p.name, p.stat().st_size)

In [ ]:
import pandas as pd, numpy as np, json, time
from pathlib import Path

DATA = Path('/content/work/data')
RUN_DIR = Path('/content/work/outputs/llm_zeroshot')

train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts_file = DATA / 'abstracts_merged_v2.csv'
if not abstracts_file.exists():
    abstracts_file = DATA / 'abstracts_merged_v3.csv'
abstracts = pd.read_csv(abstracts_file)
print('using abstract cache:', abstracts_file.name)

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract']]

def attach(df, split):
    df = df.copy()
    df['source_split'] = split
    out = df.merge(abs_map, on=['source_split', 'id'], how='left')
    out['abstract'] = out['abstract'].fillna('')
    out['has_abstract'] = out['has_abstract'].fillna(False).astype(bool)
    return out

train_full = attach(train, 'train').reset_index(drop=True)
public_full = attach(public, 'public_test').reset_index(drop=True)
private_full = attach(private, 'private_test').reset_index(drop=True)
for name, df in [('train', train_full), ('public', public_full), ('private', private_full)]:
    print(f'{name}: rows={len(df)}, has_abstract={int(df["has_abstract"].sum())} ({df["has_abstract"].mean():.1%})')

## 3. Load Qwen2.5-7B-Instruct (4-bit quantised)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
# Alternatives:
#   'Qwen/Qwen2.5-3B-Instruct'   # faster but slightly weaker
#   'mistralai/Mistral-7B-Instruct-v0.3'  # similar size, good quality

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = 'left'  # left-pad so last position is the same in batched fwd
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()
device = next(model.parameters()).device
print('loaded:', MODEL_NAME, 'on', device)

## 4. Few-shot prompt (built from L5 rubric + train examples)

In [ ]:
SYSTEM = """You are a research assistant specialized in classifying scientific papers by their relevance to Answer Set Programming (ASP) and the broader AI-symbolic agenda (neuro-symbolic AI, neural-network verification, explainable AI built on logic).

Given a paper's title and abstract, you score its ASP / AI-symbolic relevance on an integer scale 1-5.

Scoring rubric:
- 1 = Not related to ASP. Proceedings volumes, workshop summaries, generic verification / type theory / theoretical-CS papers without an ASP component.
- 2 = Loose adjacency to ASP / KR. Argumentation, description logic, default reasoning, planning, formal verification with no ASP.
- 3 = Generic logic / declarative reasoning that touches ASP vocabulary (Prolog, Datalog, abductive reasoning, NMR, qualitative reasoning) but is not ASP-centric.
- 4 = Applied / extending ASP. Papers that use ASP to solve a domain problem, extend ASP with new constructs (probabilities, choice, preferences, learning), or build tools / encodings on top of an ASP solver.
- 5 = Core ASP advances or ASP × AI cross-overs. ASP solver / grounder algorithms (Clingo, DLV, ASP(Q)), formal semantics of ASP, ASP-driven learning of programs / heuristics, neuro-symbolic systems built on ASP. Inside CAV / LICS, label 5 also covers neural-network verification and ML-meets-formal-methods work.

Examples:

Title: Proceedings 41st International Conference on Logic Programming, ICLP 2025, Rende, Italy.
Abstract: (no abstract available)
Answer: 1

Title: Synthesizing Reactive Systems from Hyperproperties.
Abstract: We present an algorithm for synthesizing reactive systems from hyperproperty specifications. We focus on a fragment of HyperLTL and provide a synthesis algorithm based on bounded synthesis.
Answer: 2

Title: DatalogMTL over the Integer Timeline.
Abstract: We study DatalogMTL, an extension of Datalog with metric temporal operators. Our main contribution is a tight characterisation of the data complexity of reasoning in the integer timeline setting.
Answer: 3

Title: An Answer Set Programming Approach to Argumentative Reasoning in the ASPIC+ Framework.
Abstract: We present an ASP encoding of argumentative reasoning in ASPIC+. The encoding is correct and complete, and we evaluate it on benchmark instances.
Answer: 4

Title: Formally Explaining Decision Tree Models with Answer Set Programming.
Abstract: We propose a formal framework for explaining decision tree predictions using Answer Set Programming. We provide both abductive and contrastive explanations and demonstrate the approach on standard benchmarks.
Answer: 5

Output ONLY a single digit 1, 2, 3, 4, or 5. Do not add any explanation, prefix, or suffix."""


def build_user_message(title: str, abstract: str) -> str:
    title = (str(title) if title is not None else '').strip()
    abstract = (str(abstract) if abstract is not None else '').strip()
    if len(abstract) > 1500:
        abstract = abstract[:1500] + '...'
    if not abstract:
        abstract = '(no abstract available)'
    return f'Title: {title}\nAbstract: {abstract}\nAnswer:'


def build_chat(title: str, abstract: str) -> list[dict]:
    return [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': build_user_message(title, abstract)},
    ]


# Sanity print: chat template length on a sample
sample_chat = build_chat(train_full.iloc[0]['title'], train_full.iloc[0]['abstract'])
sample_text = tokenizer.apply_chat_template(sample_chat, tokenize=False, add_generation_prompt=True)
print('sample prompt token length:', len(tokenizer.encode(sample_text)))
print('---')
print(sample_text[:600], '...')

## 5. Batched logit scoring (verbalizer over digits 1-5)

In [ ]:
from tqdm.auto import tqdm

# Get the token id for each digit '1'..'5' (Qwen tokenizes them as single tokens)
LABEL_TOKEN_IDS = []
for d in ['1', '2', '3', '4', '5']:
    ids = tokenizer.encode(d, add_special_tokens=False)
    if len(ids) != 1:
        # Fall back: use the first token id
        print(f'WARNING: digit {d} tokenized as {ids}')
    LABEL_TOKEN_IDS.append(ids[0])
LABEL_TOKEN_IDS = torch.tensor(LABEL_TOKEN_IDS, device=device)
print('LABEL_TOKEN_IDS =', LABEL_TOKEN_IDS.tolist())
VALUES = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0], device=device)

BATCH = 8         # T4: 8, A100: 32
MAX_LEN = 2048

@torch.no_grad()
def score_dataset(df, name):
    n = len(df)
    expected = np.zeros(n, dtype=np.float32)
    probs_full = np.zeros((n, 5), dtype=np.float32)
    titles = df['title'].tolist()
    abstracts = df['abstract'].tolist()

    for start in tqdm(range(0, n, BATCH), desc=f'scoring {name}'):
        end = min(start + BATCH, n)
        chats = [build_chat(titles[i], abstracts[i]) for i in range(start, end)]
        prompts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in chats]
        enc = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(device)
        outputs = model(**enc)
        # With left-padding, the last position of each example is the same: -1
        last_logits = outputs.logits[:, -1, :]                       # (B, vocab)
        label_logits = last_logits[:, LABEL_TOKEN_IDS]               # (B, 5)
        p = torch.softmax(label_logits.float(), dim=-1)              # (B, 5)
        score = (p * VALUES).sum(dim=-1)                             # (B,)
        expected[start:end] = score.cpu().numpy()
        probs_full[start:end] = p.cpu().numpy()
    return expected, probs_full

t0 = time.time()
train_scores, train_probs = score_dataset(train_full, 'train')
public_scores, public_probs = score_dataset(public_full, 'public')
private_scores, private_probs = score_dataset(private_full, 'private')
print(f'\nTotal scoring time: {(time.time()-t0)/60:.1f} min')
print('train scores stats:', round(train_scores.mean(), 3), '+/-', round(train_scores.std(), 3),
      'min/max', round(train_scores.min(), 3), round(train_scores.max(), 3))

## 6. Sanity check: per-true-label mean score on train

In [ ]:
from sklearn.metrics import cohen_kappa_score, mean_absolute_error

y_class = train_full['Label'].astype(int).to_numpy()
import pandas as _pd
_per_label_mean = _pd.DataFrame({'true': y_class, 'score': train_scores}).groupby('true')['score'].agg(['mean', 'std', 'count'])
print('Mean LLM score per true label (train):')
print(_per_label_mean.round(3).to_string())

# Quick sanity QWK on rounded prediction (no threshold tuning yet)
rounded = np.clip(np.round(train_scores), 1, 5).astype(int)
print('\nNo-tuning round-QWK on train:', round(cohen_kappa_score(y_class, rounded, weights='quadratic'), 4))
print('No-tuning MAE:', round(mean_absolute_error(y_class, rounded), 4))

## 7. Threshold tuning on train scores (constrained)

In [ ]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score

TRAIN_DIST = pd.Series(y_class).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()
DIST_PENALTY_LAMBDA = 0.5

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def predicted_dist(labels):
    return pd.Series(labels).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()

def tune_thresholds_constrained(y_true, scores, lambd=DIST_PENALTY_LAMBDA, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        gap_pen = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        labels = scores_to_labels(scores, thr)
        qwk = cohen_kappa_score(y_true, labels, weights='quadratic')
        dist_pen = float(np.sum(np.abs(predicted_dist(labels) - TRAIN_DIST)))
        return -qwk + gap_pen + lambd * dist_pen
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15,
                                 polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    return thr, cohen_kappa_score(y_true, scores_to_labels(scores, thr), weights='quadratic')

thresholds, oof_qwk = tune_thresholds_constrained(y_class, train_scores)
oof_pred = scores_to_labels(train_scores, thresholds)
print('Constrained-tuned OOF QWK =', round(oof_qwk, 4))
print('thresholds =', thresholds.tolist())
print('OOF predicted dist =', dict(zip([1,2,3,4,5], predicted_dist(oof_pred).round(3).tolist())))
print('TRAIN actual dist  =', dict(zip([1,2,3,4,5], TRAIN_DIST.round(3).tolist())))
print('OOF MAE  =', round(mean_absolute_error(y_class, oof_pred), 4))
print('OOF macro-F1 =', round(f1_score(y_class, oof_pred, average='macro'), 4))

## 8. Save artefacts (OOF + public + private + a standalone submission for sanity)

In [ ]:
public_pred = scores_to_labels(public_scores, thresholds)
private_pred = scores_to_labels(private_scores, thresholds)

metrics = {
    'method': 'llm_zeroshot',
    'model': MODEL_NAME,
    'quantization': '4-bit nf4 with double quant',
    'batch_size': BATCH,
    'max_length': MAX_LEN,
    'oof_qwk': float(oof_qwk),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'thresholds': [float(v) for v in thresholds],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(
        np.concatenate([public_pred, private_pred])).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
    'note': 'Continuous score = sum_d d * P(token=d|prompt) for d in 1..5. Pure zero-shot, no fold-based fitting; OOF == train scoring directly.',
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

# Save scores in same format as SPECTER2 anchors so the local stacking script picks them up.
pd.DataFrame({'id': train_full['id'], 'Label': y_class,
              'oof_score': train_scores, 'oof_pred': oof_pred,
              **{f'p_label_{i+1}': train_probs[:, i] for i in range(5)}}).to_csv(
    RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full['id'], 'score': public_scores, 'pred': public_pred,
              **{f'p_label_{i+1}': public_probs[:, i] for i in range(5)}}).to_csv(
    RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full['id'], 'score': private_scores, 'pred': private_pred,
              **{f'p_label_{i+1}': private_probs[:, i] for i in range(5)}}).to_csv(
    RUN_DIR / 'private_scores.csv', index=False)

combo = pd.concat([
    pd.DataFrame({'id': public_full['id'], 'Label': public_pred}),
    pd.DataFrame({'id': private_full['id'], 'Label': private_pred}),
], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'llm_zeroshot_submission.csv', index=False)
print('submission rows =', len(submission))
print(submission.head())

## 9. Zip + download

Extract into local `outputs/llm_zeroshot/` and re-run `python train_stacking_meta.py`. The local script does not yet auto-detect this anchor — see the next section after download.

In [ ]:
zip_path = pathlib.Path('/content/llm_zeroshot_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.iterdir():
        zf.write(p, arcname=f'llm_zeroshot/{p.name}')
print('zipped:', zip_path, 'size MB =', round(zip_path.stat().st_size / 1e6, 2))
files.download(str(zip_path))